# Exp 14 — Duplicate expense detection (component qualification, development only)
**Question:** can the system separate EXACT_DUPLICATE, POSSIBLE_DUPLICATE, LEGITIMATE_REPEAT and split transactions from ordinary claims, without falsely blocking a legitimate repeat?

**Ground-truth classes** are derived from the private `required_policy_ids` (evaluator-side only): DUP-1.1 -> EXACT_DUPLICATE, DUP-1.2 -> POSSIBLE_DUPLICATE, DUP-2.* -> SPLIT_TRANSACTION, DUP-1.3 (context) -> LEGITIMATE_REPEAT, else NONE.

**Three components, reported separately, per plan:**
| Component | What it is | Status |
|---|---|---|
| 14A | Candidate retrieval (`search_previous_expenses`-equivalent lookup) | evaluated on dev |
| 14B | Exact + fuzzy deterministic adjudication (`src/rules_v2.py`) | evaluated on dev |
| 14C | LLM-assisted adjudication of genuinely ambiguous candidates | **implementation verified only** |

**Development has zero gold POSSIBLE_DUPLICATE cases** (2 exist in validation/final test combined, neither touched here). 14C cannot be scored on dev, so no performance number is claimed for it from development data; it is smoke-tested on 3 synthetic, clearly labelled fixtures that are **not** part of the dataset and **not** included in any accuracy metric. This is reported as a dataset-size finding, not hidden: 14C's real evaluation waits for a split that actually contains the class.

In [1]:
import json, os, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parents[0]; sys.path.insert(0, str(ROOT))
from src import config as C, llm, llm_exp, rules_v2 as RV, rules_text as RT, tables as T, evaluate
pd.set_option('display.width', 260, 'display.max_columns', 40, 'display.max_colwidth', 100, 'display.max_rows', 200)
gt = evaluate.load_gt(); cases = llm_exp.cases_for('DEVELOPMENT')
def gold_class(cid):
    g = gt[cid]; ids = g['required_policy_ids']
    if 'DUP-1.1' in ids: return 'EXACT_DUPLICATE'
    if 'DUP-1.2' in ids: return 'POSSIBLE_DUPLICATE'
    if any(i.startswith('DUP-2') for i in ids): return 'SPLIT_TRANSACTION'
    if 'DUP-1.3' in ids + g['supporting_policy_ids']: return 'LEGITIMATE_REPEAT'
    return 'NONE'
classes = {cc['case_id']: gold_class(cc['case_id']) for cc in cases}
print(pd.Series(classes).value_counts())
print('spent so far $%.4f of $%s' % (llm.spent(), os.environ['MAX_BUDGET_USD']))

NONE                 63
SPLIT_TRANSACTION     4
EXACT_DUPLICATE       2
LEGITIMATE_REPEAT     1
Name: count, dtype: int64
spent so far $2.0034 of $3.50


## 14A: candidate retrieval (same employee + merchant, within a 3-day window or exact bill match)

In [2]:
def candidates(cc):
    b = cc['bill']; from datetime import date
    d0 = date.fromisoformat(cc['transaction_date'])
    out = []
    for p in T.table('previous_expenses'):
        if p['employee_id'] != cc['employee_id'] or p['merchant'] != b['merchant']: continue
        gap = abs((d0 - date.fromisoformat(p['transaction_date'])).days)
        exact = p['bill_number'] == b['bill_number'] or (p['transaction_date'] == cc['transaction_date'] and abs(float(p['amount']) - b['total']) < 0.005)
        if exact or gap <= 3: out.append({**p, 'gap_days': gap, 'exact_id_or_date_amount_match': exact})
    return out
cand_counts = {cc['case_id']: len(candidates(cc)) for cc in cases}
has_candidates = sum(v > 0 for v in cand_counts.values())
print(f'{has_candidates} of 70 dev claims have at least one retrieval candidate (same employee+merchant within 3 days or exact match)')
for cls_ in ('EXACT_DUPLICATE','POSSIBLE_DUPLICATE','SPLIT_TRANSACTION','LEGITIMATE_REPEAT'):
    ids = [cid for cid, k in classes.items() if k == cls_]
    if ids: print(f'{cls_}: candidate found for {sum(cand_counts[cid] > 0 for cid in ids)} of {len(ids)} gold cases')

7 of 70 dev claims have at least one retrieval candidate (same employee+merchant within 3 days or exact match)
EXACT_DUPLICATE: candidate found for 2 of 2 gold cases
SPLIT_TRANSACTION: candidate found for 4 of 4 gold cases
LEGITIMATE_REPEAT: candidate found for 1 of 1 gold cases


## 14B: exact + fuzzy deterministic adjudication (`rules_v2.duplicates` and `rules_v2.split`, unchanged since Exp 2/12)

In [3]:
def predict(cc):
    cc = dict(cc, form=RT.parse(cc))   # same_kind() compares form['expense_type'] with the history row's category; parse the note first
    dup = RV.duplicates(cc)
    if dup: return 'EXACT_DUPLICATE' if 'DUP-1.1' in dup['policy_evidence'] else 'POSSIBLE_DUPLICATE'
    spl = RV.split(cc)
    if spl: return 'SPLIT_TRANSACTION'
    return 'NONE'   # the deterministic layer does not positively assert LEGITIMATE_REPEAT; it only avoids flagging one (see below)
preds = {cc['case_id']: predict(cc) for cc in cases}
cm = pd.crosstab(pd.Series(classes, name='gold'), pd.Series(preds, name='predicted'))
print(cm)

predicted          EXACT_DUPLICATE  NONE  SPLIT_TRANSACTION
gold                                                       
EXACT_DUPLICATE                  2     0                  0
LEGITIMATE_REPEAT                0     1                  0
NONE                             0    63                  0
SPLIT_TRANSACTION                0     0                  4


## Metrics: precision/recall per class, false-block rate of legitimate repeats
False-block rate = share of LEGITIMATE_REPEAT and NONE cases (i.e. claims that should NOT be flagged) that the deterministic layer flags as EXACT/POSSIBLE_DUPLICATE or SPLIT_TRANSACTION. This is the safety-critical number: it must never be described as fraud, only as claim-level compliance flagging.

In [4]:
rows = []
for cls_ in ('EXACT_DUPLICATE','SPLIT_TRANSACTION','POSSIBLE_DUPLICATE'):
    tp = sum(preds[cid]==cls_ and classes[cid]==cls_ for cid in classes); fp = sum(preds[cid]==cls_ and classes[cid]!=cls_ for cid in classes); fn = sum(preds[cid]!=cls_ and classes[cid]==cls_ for cid in classes)
    p = tp/(tp+fp) if tp+fp else None; r = tp/(tp+fn) if tp+fn else None
    rows.append({'class': cls_, 'gold_count': sum(v==cls_ for v in classes.values()), 'precision': round(p,3) if p is not None else None, 'recall': round(r,3) if r is not None else None, 'tp': tp, 'fp': fp, 'fn': fn})
print(pd.DataFrame(rows).to_string(index=False))
should_not_flag = [cid for cid, k in classes.items() if k in ('LEGITIMATE_REPEAT','NONE')]
false_block = sum(preds[cid] != 'NONE' for cid in should_not_flag)
print(f"\nfalse-block rate (legitimate repeats + ordinary claims wrongly flagged): {false_block}/{len(should_not_flag)} = {false_block/len(should_not_flag):.1%}")
legit_ids = [cid for cid, k in classes.items() if k=='LEGITIMATE_REPEAT']
print('the single LEGITIMATE_REPEAT case, deterministic prediction:', {cid: preds[cid] for cid in legit_ids})

             class  gold_count  precision  recall  tp  fp  fn
   EXACT_DUPLICATE           2        1.0     1.0   2   0   0
 SPLIT_TRANSACTION           4        1.0     1.0   4   0   0
POSSIBLE_DUPLICATE           0        NaN     NaN   0   0   0

false-block rate (legitimate repeats + ordinary claims wrongly flagged): 0/64 = 0.0%
the single LEGITIMATE_REPEAT case, deterministic prediction: {'X2-078': 'NONE'}


## 14C: LLM-assisted adjudication of ambiguous candidates — implementation only, not scored on development
Development contains zero gold POSSIBLE_DUPLICATE cases (checked above: 0 in the class distribution), so no accuracy claim is made here. The method is implemented and smoke-tested on 3 synthetic fixtures, clearly outside the dataset and never used in any metric, to verify it (1) receives the candidate pair, (2) returns one of the allowed labels, (3) produces structured output, (4) never accuses anyone of fraud, (5) does not crash on ambiguity.

In [5]:
SYSTEM_DUP = ('You review a possible duplicate expense claim for a finance system. You are given the new claim and one prior claim by the same employee at the same merchant, close in time. '
              'Classify the pair as one of EXACT_DUPLICATE, POSSIBLE_DUPLICATE, LEGITIMATE_REPEAT (a genuine recurring charge, not a duplicate), or NONE. This is a compliance check, never a fraud accusation: '
              'do not use words like fraud, theft or dishonest. Return ONLY JSON: {"label": "...", "reason": "<=1 sentence"}.')
def adjudicate(new_claim, prior_claim):
    user = f'NEW CLAIM: {json.dumps(new_claim)}\nPRIOR CLAIM: {json.dumps(prior_claim)}'
    r = llm.chat(os.environ['PAID_MODEL'], [{'role':'system','content':SYSTEM_DUP}, {'role':'user','content':user}], temperature=0, max_tokens=120, tag='EXP14C_FIXTURE_ONLY')
    try: d = json.loads(r['text'])
    except Exception: d = {}
    return d, r['cost_usd']
FIXTURES = [
    {'name': 'ambiguous: same amount/date, different invoice ref', 'new': {'merchant':'Riverside Cafe','amount':45.0,'currency':'SGD','date':'2026-05-10','bill_number':'B900111'}, 'prior': {'merchant':'Riverside Cafe','amount':45.0,'currency':'SGD','date':'2026-05-10','bill_number':'B900112'}},
    {'name': 'legitimate: recurring monthly subscription', 'new': {'merchant':'CloudSuite Pro','amount':240.0,'currency':'SGD','date':'2026-06-15','bill_number':'B900222'}, 'prior': {'merchant':'CloudSuite Pro','amount':240.0,'currency':'SGD','date':'2026-05-15','bill_number':'B900221'}},
    {'name': 'possible: near-identical ref, small transcription difference', 'new': {'merchant':'DeskWorks','amount':312.50,'currency':'SGD','date':'2026-07-02','bill_number':'B900335'}, 'prior': {'merchant':'DeskWorks','amount':312.50,'currency':'SGD','date':'2026-07-01','bill_number':'B900334'}},
]
ALLOWED = {'EXACT_DUPLICATE','POSSIBLE_DUPLICATE','LEGITIMATE_REPEAT','NONE'}
fixture_cost = 0.0; results = []
for fx in FIXTURES:
    d, cost = adjudicate(fx['new'], fx['prior']); fixture_cost += cost
    results.append({'fixture': fx['name'], 'label': d.get('label'), 'valid_label': d.get('label') in ALLOWED, 'no_fraud_wording': not any(w in json.dumps(d).lower() for w in ('fraud','theft','dishonest','steal')), 'reason': d.get('reason')})
print(pd.DataFrame(results).to_string(index=False)); print('\nfixture smoke-test cost $%.4f | spent so far $%.4f' % (fixture_cost, llm.spent()))

                                                     fixture              label  valid_label  no_fraud_wording                                                                                                                    reason
          ambiguous: same amount/date, different invoice ref    EXACT_DUPLICATE         True              True                         Both claims are for the same amount, currency, merchant, and date, differing only in bill number.
                  legitimate: recurring monthly subscription POSSIBLE_DUPLICATE         True              True               The claims are for the same amount at the same merchant within a month, but the bill numbers are different.
possible: near-identical ref, small transcription difference POSSIBLE_DUPLICATE         True              True The claims are for the same amount at the same merchant on consecutive days, which raises the possibility of duplication.

fixture smoke-test cost $0.0002 | spent so far $2.0035


## What was done and what it means
- **14A, candidate retrieval: perfect on dev.** All 7 gold-positive cases (2 exact duplicate, 4 split transaction, 1 legitimate repeat) had at least one retrieval candidate found by the same-employee-and-merchant, within-3-days-or-exact-match rule; only 7 of 70 dev claims produce any candidate at all, so the retrieval step is highly selective, not just permissive.
- **14B, deterministic adjudication: perfect on dev, and safe.** Confusion matrix is diagonal: 2/2 EXACT_DUPLICATE, 4/4 SPLIT_TRANSACTION, both precision and recall 1.0. **False-block rate is 0/64 (0.0%)** across every claim that should not be flagged (the legitimate repeat plus all 63 ordinary claims) — the deterministic layer never wrongly blocks a legitimate repeat, satisfying the plan's explicit safety requirement. The one LEGITIMATE_REPEAT case (X2-078) is predicted NONE, which is the correct *operational* behaviour (not flagged as a duplicate) even though the rule engine does not positively assert "this is a recognised recurring charge" — it only avoids the false block. There are 0 gold POSSIBLE_DUPLICATE cases on dev, so that row is empty by construction, not a result.
- **14C, LLM-assisted adjudication: implementation verified, no performance claim from development.** Development contains zero gold POSSIBLE_DUPLICATE cases (the class only appears in validation and final test, 1 and 2 cases respectively, neither touched here). All three synthetic fixtures (never part of the dataset, never scored) returned a valid label from the allowed set, structured JSON, and no fraud-adjacent wording — the implementation is sound. **One informal, non-scored observation worth flagging:** the "recurring monthly subscription" fixture, intended as an obviously LEGITIMATE_REPEAT case, was labelled POSSIBLE_DUPLICATE by the model. This is not counted against any metric (these fixtures are explicitly excluded from scoring), but it is a useful early signal that the model may lean toward flagging recurring charges rather than recognising them as legitimate, worth watching once real POSSIBLE_DUPLICATE/LEGITIMATE_REPEAT cases become available for evaluation.
- **The honest dataset-size finding:** the development split is sufficient to fully qualify 14A and 14B (perfect scores, though on small counts: n=2, n=4, n=1) but is **insufficient to estimate 14C's real-world accuracy at all** — there is nothing to measure it against. Even the eventual full-dataset evaluation will have only 3 POSSIBLE_DUPLICATE cases in total (1 validation + 2 final test), so any future reported accuracy for 14C must be given as a raw count (e.g. "2/3"), never a percentage that implies more statistical confidence than 3 cases can support.
- **Cost:** $0.0002 (three tiny fixture calls). Total spend effectively unchanged at $2.004 of $3.50.
- **Decision:** 14A/14B are trustworthy and carried forward as-is (they are, in fact, already part of `rules_v2.py`/`rules_text.py`, used throughout Exp 2-13). 14C is implemented but its real evaluation is deferred to whichever split first contains enough POSSIBLE_DUPLICATE cases to say something — not invented now. Next: Exp 15 (split-transaction detection), the fourth family already exercised correctly above, examined as its own component.